# Sliding Dot product, (Circular) Convolution, and Overlap-Add!

In [1]:
import numpy as np
import scipy
import time

## 1. Introduction

This tutorial shows how [Convolution](https://en.wikipedia.org/wiki/Convolution) can be used to compute sliding-dot-product (sdp). The sdp between two 1D arrays, Q (query) and T (a time series), is 1D array with length `len(T) - len(Q) + 1`, and its `i-th` element is: `np.dot(Q, T[i: i + len(Q)])`. An example is provided below:


Q = [A, B]
<br>
T = [1, 2, 3, 4]

Their sdp is: <br>
SDP(Q, T) = [1*A + 2*B, 2*A + 3*B, 3*A + 4*B]


## 2. Linear Convolution

It can be shown that sdp can be computed via convolution. The general formula of [the discrete convolution](https://en.wikipedia.org/wiki/Convolution#Discrete_convolution) between two signals is as follows:

$$ (x * h)[i] = \sum_{j=-\infty}^{j=+\infty}{x[j]h[i-j]}  $$


In our case, we are working with signals with finite legnths, meaning the values of an out-of-range index is zero. Let's try this for our example. Note that for a given index `i`, `h[i-j]` is moving backward as `j` increases. In other words, convolution reverse one of the signals. Therefore, we also flip `Q` before applying convolution to it! So when it is flipped again during convolution, it gives the correct sliding dot product.

Q = [A, B] 
<br>
Q_flip = [B, A] --> h
<br>
T = [1, 2, 3, 4] --> x

Let's compute the convolution between x and h:
* ...
* i = -2 --> $(x*h)[-2] = \sum_{j=-\infty}^{j=+\infty}{x[j]h[-2-j]} = 0$
* i = -1 --> $(x*h)[-1] = \sum_{j=-\infty}^{j=+\infty}{x[j]h[-1-j]} = 0$
* i = 0 --> $(x*h)[0] = \sum_{j=-\infty}^{j=+\infty}{x[j]h[0-j]} = x[0]h[0] = 1B$
* i = 1 --> $(x*h)[1] = \sum_{j=-\infty}^{j=+\infty}{x[j]h[1-j]} = x[0]h[1] + x[1]h[0] = 1A + 2B$
* i = 2 --> $(x*h)[1] = \sum_{j=-\infty}^{j=+\infty}{x[j]h[2-j]} = x[1]h[1] + x[2]h[0] = 2A + 3B$
* i = 3 --> $(x*h)[1] = \sum_{j=-\infty}^{j=+\infty}{x[j]h[3-j]} = x[2]h[1] + x[3]h[0] = 3A + 4B$
* i = 4 --> $(x*h)[1] = \sum_{j=-\infty}^{j=+\infty}{x[j]h[4-j]} = x[3]h[1] + x[4]h[0] = 4A$
* i = 5 --> $(x*h)[1] = \sum_{j=-\infty}^{j=+\infty}{x[j]h[5-j]} = 0 $
* i = 6 --> $(x*h)[1] = \sum_{j=-\infty}^{j=+\infty}{x[j]h[6-j]} = 0 $
* ...

The actual values are at indices in `range(0, 5)`, i.e. [1B, 1A+2B, 2A+3B, 3A+4B, 4A]. 

**A couple of notes here:**
<br>
(1) The length of output signal is `5 = len(T) + len(Q) - 1`. 
<br>
(2) The sdp of Q and T, i.e. [1A + 2B, 2A + 3B, 3A + 4B], is in the slice `[len(Q)-1 : len(T)]`.

In [2]:
# Let's check our example for A=0 and B=1
Q = np.array([0, 1])
T = np.array([1, 2, 3, 4])

QT_conv = scipy.signal.convolve(T, Q[::-1], mode='full')
print(QT_conv)

[1 2 3 4 0]


Now that we have the "full" output, we can slice it for the range `[len(Q) - 1 : len(T)]` to get the sdp: [2, 3, 4]. This is known as the ["valid" mode](https://github.com/numpy/numpy/blob/c5ab79c14c98bfda1e60770ffa23a6130f8267b7/numpy/_core/numeric.py#L835-L839):

> The convolution product is only given for points where the signals overlap completely.

In [3]:
QT_conv_valid = scipy.signal.convolve(T, Q[::-1], mode='valid')
print(QT_conv_valid)

[2 3 4]


And that is the sdp of Q and T!

## 3. Circular Convolution

[Circular Convolution](https://en.wikipedia.org/wiki/Circular_convolution) is the convolution of two periodic functions that have the same period, $N$. Their convolution has length $N$, and it can be computed as follows:

$$CONV_{x,h}[i] = \sum_{j=0}^{N-1} x[j] \cdot h[(i - j) \mod N]$$

Recall our initial example:

Q = [A, B]
<br>
Q_flip = [B, A] --> h
<br>
T = [1, 2, 3, 4] --> x


Let's use circular convolution for the two following cases:

**Case 1:** <br>
Pad both arrays $x$ and $h$ with enough zeros till the length of each array becomes `N = len(x) + len(h) - 1`


In this case, `N = 4 + 2 - 1 = 5`. Therefore: <br>

h_padded = [B, A, 0, 0, 0] <br>
x_padded = [1, 2, 3, 4, 0]

Thier circular convolution is: [1B, 1A + 2B, 2A + 3B, 3A + 4B, 4A]. <br>
This is the same full linear convolution we achieved before!



**Case 2:** <br>
Pad both arrays $x$ and $h$ with enough zeros till the length of each array becomes `max(len(x), len(h))`.

In this cae, `N = max(4, 2) = 4`. Therefore: <br>

h_padded = [B, A, 0, 0] <br>
x = [1, 2, 3, 4]

Thier circular convolution is: [1B + 3A, 1A + 2B, 2A + 3B, 3A + 4B]
<br>
Although this is NOT the full linear convolution, it has the `sdp` piece, located in `range(2-1:4)`

In [4]:
def naive_sdp(Q, T):
    m = len(Q)
    n = len(T)
    l = n - m + 1
    out = np.empty(l, dtype=np.float64)
    for i in range(l):
        out[i] = np.dot(Q, T[i:i+m])
    return out
    
def naive_linear_convolution(Q, T, mode='full'):
    """
    Return the convolution for the given mode.
    """
    m = len(Q)
    n = len(T)
    N = n + m - 1
    Q_padded = np.pad(Q, (0, N - m), 'constant')
    T_padded = np.pad(T, (0, N - n), 'constant')

    out = np.empty(N, dtype=np.float64)
    for i in range(N):
        out[i] = 0
        for j in range(N):
            out[i] += T_padded[j] * Q_padded[(i-j) % N]

    if mode == 'full':
        pass
    elif mode == 'valid':
        out = out[m-1: n]
    else:
        raise ValueError

    return out

In [5]:
Q = np.random.rand(10)
T = np.random.rand(40)
sdp = naive_sdp(Q, T)

# full convolution
ref_conv_full = naive_linear_convolution(Q[::-1], T, mode='full')
comp_conv_full = scipy.signal.convolve(Q[::-1], T, mode='full', method='direct')
np.testing.assert_allclose(ref_conv_full, comp_conv_full)

# valid convolution
ref_conv_valid = naive_linear_convolution(Q[::-1], T, 'valid')
comp_conv_valid = scipy.signal.convolve(Q[::-1], T, mode='valid', method='direct')
np.testing.assert_allclose(ref_conv_valid, comp_conv_valid)

# Check sdp
np.testing.assert_allclose(ref_conv_valid, sdp)

## 4. FFt-based [Circular] Convolution

The time complexity of sdp is $\mathcal{O}{(nm)}$, where `n=len(T)` and `m=len(Q)`. As mentioned in the previous sections, another way to compute sdp is to perform convolution. However, the time complexity is not lower when convolution is performed in time domain. It turns out one can use FFT to calculate circular convolution with the lower time complexiity of $\mathcal{O}{(nlog(n))}$ as follows: 

$$CONV_{x,h} = IFFT(FFT(x) * FFT(h))$$

where `x` and `h` are two arrays with same size. And if they are real-valued, we can use the real version of (I)FFT as follows:

$$CONV_{x,h} = IRFFT(RFFT(x) * RFFT(h))$$

Let's verify the result for an example:

In [6]:
def fft_convolution(Q, T, mode='full'):
    m = len(Q)
    n = len(T)
    N = m + n - 1
    Q_padded = np.pad(Q, (0, N - m), 'constant')
    T_padded = np.pad(T, (0, N - n), 'constant')

    out = scipy.fft.irfft(scipy.fft.rfft(Q_padded) * scipy.fft.rfft(T_padded), n=N)
    if mode == 'full':
        pass
    elif mode == 'valid':
        out = out[m - 1: n]
    else:
        raise ValueError(f"The mode `{mode}` is not supported")
    
    return out

In [7]:
Q = np.random.rand(10)
T = np.random.rand(40)
sdp = naive_sdp(Q, T)

# full convolution
ref_conv_full = naive_linear_convolution(Q[::-1], T, mode='full')
comp_conv_full = fft_convolution(Q[::-1], T, mode='full')
np.testing.assert_allclose(ref_conv_full, comp_conv_full)

# valid convolution
ref_conv_valid = naive_linear_convolution(Q[::-1], T, 'valid')
comp_conv_valid = fft_convolution(Q[::-1], T, mode='valid')
np.testing.assert_allclose(ref_conv_valid, comp_conv_valid)

# Again, recall that valid here is equivalent to sdp
np.testing.assert_allclose(ref_conv_valid, sdp)

Note that, from Sliding-Dot-Product's standpoint, we only care about the 'valid' mode. We do not need to pad Q and T to increase their lengths to `len(Q) + len(T) - 1`. We can actually just pad the shorter array, the query Q, till its length becomes the same as `len(T)`. And then we can perform circular convolution. See **Case 2** provided in **3. Circular Convolution**. This should be faster as RFFT / IRFFT will be applied on arrays with length `len(T)` instead of `len(Q) + len(T) - 1`.

In [8]:
def valid_fft_convolution(Q, T):
    m = len(Q)
    n = len(T)
    Q_padded = np.pad(Q, (0, n - m), 'constant')

    return scipy.fft.irfft(scipy.fft.rfft(Q_padded) * scipy.fft.rfft(T), n=n)[m -1 : n]

In [9]:
Q = np.random.rand(10)
T = np.random.rand(40)
sdp = naive_sdp(Q, T)

ref_conv_valid = naive_linear_convolution(Q[::-1], T, 'valid')
comp_conv_valid = valid_fft_convolution(Q[::-1], T)
np.testing.assert_allclose(ref_conv_valid, comp_conv_valid)

# check sdp
np.testing.assert_allclose(sdp, ref_conv_valid)

As the last part of this section, let's use scipy's API to perform the FFT-based convoluton.

In [10]:
Q = np.random.rand(10)
T = np.random.rand(40)
sdp = naive_sdp(Q, T)

ref_QT_convolve = valid_fft_convolution(Q[::-1], T)
comp_QT_convolve = scipy.signal.fftconvolve(Q[::-1], T, mode='valid')
np.testing.assert_allclose(ref_QT_convolve, comp_QT_convolve)

## 5. Compare the Performances: `scipy.signal.fftconvolve` vs `valid_convolution`

In the last section, we mentioned that, from sliding-dot-product's standpoint, we can just pad the shorter array with enough zeros till its length becomes the same as the longer array. We claimed that its performance should be faster than when both arrays are padded with zeros till its length becomes the same as the output of full convolution, which is what [scipy.signal.fftconvolve](https://github.com/scipy/scipy/blob/b1296b9b4393e251511fe8fdd3e58c22a1124899/scipy/signal/_signaltools.py#L727) does. Let's check their performances for an example:

In [11]:
timeout = 60

T = np.random.rand(2 ** 20)
Q = np.random.rand(2 ** 19)

##########
# scipy.signal.fftconvolve
total_time = 0
n_iter = 0
ref = scipy.signal.fftconvolve(Q[::-1], T, mode='valid')
while total_time < timeout:
    start = time.time()
    scipy.signal.fftconvolve(Q[::-1], T, mode='valid')
    total_time += time.time() - start
    n_iter += 1
ref_timing = total_time / n_iter

##########
# valid_fft_convolution
total_time = 0
n_iter = 0
comp = valid_fft_convolution(Q[::-1], T)
while total_time < timeout:
    start = time.time()
    valid_fft_convolution(Q[::-1], T)
    total_time += time.time() - start
    n_iter += 1
comp_timing = total_time / n_iter

##########
# assert outputs
np.testing.assert_allclose(ref, comp)

# Check performance
performance_ratio = ref_timing / comp_timing
print(f'valid_fft_convolution is {performance_ratio} times faster than scipy.signal.fftconvolve, which computes `full` convolution')

valid_fft_convolution is 1.828499420419656 times faster than scipy.signal.fftconvolve, which computes `full` convolution


Note that our `valid_fft_convolution` function is about 80% faster in this case. The query `Q` was purposefully set to a large array so that its share in `len(T) + len(Q) - 1` becomes considerable, and its impact on performance becomes clearer.

## 6. Overlap-add Convolution

So far, we learned that we can use FFT-based convolution to perform circular convolution on large arrays to speed-up the calculation of sliding-dot-product. Can we do better? We can! It turns out the circular convolution can be achieved by performing several circular convolution on smaller arrays. The [overlap-add method](https://en.wikipedia.org/wiki/Overlap–add_method) takes this approach. It breaks down the data into non-overlapping chunks and then sum (add) tail of one block with head of the next block (overlap). 

<br>
To better understand this, we can use the following example to see how circular convolution can be broken down into smaller pieces.

**Example:**

Q = [A, B] 
<br>
T = [1, 2, 3, 4, 5, 6]

sdp is: <br>
sdp(Q, T) = [1A + 2B, 2A + 3B, 3A + 4B, 4A + 5B, 5A + 6B]

What if I break down T into two non-overlapping chunks `T1=[1, 2, 3]` and `T2=[4, 5, 6]`? Let's compute the sdp for each:


sdp(Q, T1) = [1A + 2B, 2A + 3B] <br>
sdp(Q, T2) = [4A + 5B, 5A + 6B]

If I put these elements together, I can see that one of the elements of sdp, `3A + 4B`, is missing. This is because breaking down the time series into NON-overlapping chunks destroyed one of the subsequences in this example. Is there a way to not lose that element of sdp? We need to find a way to construct that element. `3A + 4B` can be written as: `(3A + 0B) + (0A + 4B)`. If we look closely, we can see:

* The first component, `3A + 0B` is the dot product of `[3, 0]` and `[A, B]` 
* The second component, `0A + 4B` is the dot product of `[0, 4]` and `[A, B]`

So, what if we pad each of those two chunks with 0?

```
T1' = [1, 2, 3, 0] --> sdp(Q, T1) = [1A + 2B, 2A + 3B, 3A + 0B]
T2' = [0, 4, 5, 6] --> sdp(Q, T1) = [0A + 4B, 4A + 5B, 5A + 6B]
```

And now I can add the last element of `sdp(Q, T1)` to the first element of `sdp(Q, T2)` to get that missing element`3A + 4B`!

Now, we will show that this approach still works if we append `0` to the end of `T2`, and use circular convolution!

```
# chunk T, and append `len(Q)-1` zeros to each
T1' = [1, 2, 3, 0]  
T2' = [4, 5, 6, 0] 

# flip Q and pad it with zeros to get the same block_size as T1', T2'
Q' =  [B, A, 0, 0]    
```

**and their corresponding circular convolutions are:**

* circular_convolution(Q', T1'): [1B, 1A + 2B, 2A + 3B, **3A + 0B**]
* circular_convolution(Q', T1'): [**0A + 4B**, 4A + 5B, 5A+6B, 6A + 0B]



The last element of each convolution is meaningless here unless it is added to the first element of the convolution that corresponds to the next chunk. Therefore, we can:

(1) Slice the output upto a point where zero-padding started
<br>
(2) add last element of convolution of a chunk to the first element of convolution of next chunk
<br>
(3) Put the elements together, flatten the result, and get the slice `[M - 1 : N]`

And this will give us the sdp of Q and T: <br>
`[1A + 2B, 2A + 3B, (3A + 0B) + (0A + 4B), 4A + 5B, 5A + 6B]`


**In general, the algorith for overlap-add is as follows:**

* Choose a `block_size`: this is the size of array to which the circular convolution will be applied.
* Based on the `block_sise`, find the `chunk_size = block_size - (m - 1)`
* Break down `T` into non-overlapping chunks, each with size `chunk_size`, and pad each chunk with `m-1` zeros so that the lenght becomes `block_size`
* Calculate circular convolution between each block and `Q[::-1]` (via FFT approach), and use the overlap-add logic to reconstruct the circular convolution of full array to obtain the 'valid' convolution.